# Lab - Phân loại nhiều lớp


## 1.1 Bàn thắng
Trong Lab này, bạn sẽ khám phá một ví dụ về phân loại nhiều lớp bằng cách sử dụng neural networks.
<figure>
 <img src="./images/C2_W2_mclass_header.png"   style="width500px;height:200px;">
</figure>


##1.2 Công cụ
Bạn sẽ sử dụng một số thói quen vẽ đồ thị. Chúng được lưu trữ trong `lab_utils_multiclass_TF.py` trong folder này.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
np.set_printoptions(precision=2)
from lab_utils_multiclass_TF import *
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

#2.0 Phân loại nhiều lớp
Mạng thần kinh thường được sử dụng để phân loại dữ liệu. Ví dụ là neural network:
- chụp ảnh và phân loại đối tượng trong ảnh là {chó,mèo,ngựa,khác}
- lấy một câu và phân loại 'các thành phần của lời nói' trong các thành phần của nó: {danh từ, động từ, tính từ, v.v..}  

Một mạng loại này sẽ có nhiều đơn vị ở lớp cuối cùng. Mỗi đầu ra được liên kết với một danh mục. Khi một ví dụ đầu vào được áp dụng cho mạng, đầu ra có giá trị cao nhất là danh mục được dự đoán. Nếu đầu ra được áp dụng cho hàm softmax thì đầu ra của softmax sẽ cung cấp xác suất của đầu vào nằm trong mỗi danh mục. 

Trong Lab này, bạn sẽ thấy một ví dụ về cách xây dựng mạng nhiều lớp trong Tensorflow. Sau đó chúng ta sẽ xem neural network đưa ra dự đoán như thế nào.

Hãy bắt đầu bằng cách tạo một tập dữ liệu bốn lớp.


## 2.1 Chuẩn bị và trực quan hóa dữ liệu của chúng ta
Chúng ta sẽ sử dụng hàm Scikit-Learn `make_blobs` để tạo tập dữ liệu training với 4 danh mục như trong biểu đồ bên dưới.


In [2]:
# tạo tập dữ liệu 4 lớp để phân loạiclasses = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0
X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std,random_state=30)

In [3]:
plt_mc(X_train,y_train,classes, centers, std=std)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Mỗi dấu chấm đại diện cho một traning example. Trục (x0,x1) là đầu vào và màu đại diện cho lớp mà ví dụ được liên kết. Sau khi được training, mô hình sẽ đưa ra một ví dụ mới (x0,x1) và sẽ dự đoán lớp.  

Trong khi được tạo, tập dữ liệu này đại diện cho nhiều vấn đề phân loại trong thế giới thực. Có một số feature đầu vào (x0,...,xn) và một số loại đầu ra. Mô hình được training để sử dụng các feature đầu vào để dự đoán loại đầu ra chính xác.


In [4]:
# hiển thị các lớp trong tập dữ liệuprint(f"unique classes {np.unique(y_train)}")
# chỉ ra cách các lớp được thể hiệnprint(f"class representation {y_train[:10]}")
# hiển thị hình dạng của tập dữ liệu của chúng taprint(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

unique classes [0 1 2 3]
class representation [3 3 3 0 3 3 3 3 2 0]
shape of X_train: (100, 2), shape of y_train: (100,)


## 2.2 Mô hình
<img align="Right" src="./images/C2_W2_mclass_lab_network.PNG"  style=" width:350px; padding: 10px 20px ; ">
Lab này sẽ sử dụng mạng 2 lớp như được hiển thị.
Không giống như các mạng phân loại nhị phân, mạng này có bốn đầu ra, một đầu ra cho mỗi lớp. Cho một ví dụ đầu vào, đầu ra có giá trị cao nhất là loại đầu vào được dự đoán.   

Dưới đây là một ví dụ về cách xây dựng mạng này trong Tensorflow. Lưu ý rằng lớp đầu ra sử dụng kích hoạt `linear` thay vì kích hoạt `softmax`. Mặc dù có thể bao gồm softmax trong lớp đầu ra, nhưng sẽ ổn định hơn về mặt số nếu các đầu ra tuyến tính được chuyển đến hàm loss mát trong quá trình training. Nếu mô hình được sử dụng để dự đoán xác suất thì softmax có thể được áp dụng tại thời điểm đó.


In [5]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        Dense(2, activation = 'relu',   name = "L1"),
        Dense(4, activation = 'linear', name = "L2")
    ]
)

Các câu lệnh dưới đây biên dịch và training mạng. Việc đặt `from_logits=True` làm đối số cho hàm loss sẽ chỉ định rằng kích hoạt đầu ra là tuyến tính chứ không phải softmax.


In [6]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(
    X_train,y_train,
    epochs=200
)

Epoch 1/200
4/4 [==============================] - 0s 1ms/step - loss: 1.8158
Epoch 2/200
4/4 [==============================] - 0s 1ms/step - loss: 1.6976
Epoch 3/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5989
Epoch 4/200
4/4 [==============================] - 0s 961us/step - loss: 1.5179
Epoch 5/200
4/4 [==============================] - 0s 1ms/step - loss: 1.4369
Epoch 6/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3756
Epoch 7/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3154
Epoch 8/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2621
Epoch 9/200
4/4 [==============================] - 0s 991us/step - loss: 1.2188
Epoch 10/200
4/4 [==============================] - 0s 983us/step - loss: 1.1791
Epoch 11/200
4/4 [==============================] - 0s 974us/step - loss: 1.1446
Epoch 12/200
4/4 [==============================] - 0s 984us/step - loss: 1.1129
Epoch 13/200
4/4 [==============================] -

4/4 [==============================] - 0s 1ms/step - loss: 0.1758
Epoch 104/200
4/4 [==============================] - 0s 956us/step - loss: 0.1709
Epoch 105/200
4/4 [==============================] - 0s 977us/step - loss: 0.1662
Epoch 106/200
4/4 [==============================] - 0s 958us/step - loss: 0.1616
Epoch 107/200
4/4 [==============================] - 0s 952us/step - loss: 0.1575
Epoch 108/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1527
Epoch 109/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1480
Epoch 110/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1439
Epoch 111/200
4/4 [==============================] - 0s 996us/step - loss: 0.1396
Epoch 112/200
4/4 [==============================] - 0s 987us/step - loss: 0.1357
Epoch 113/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1315
Epoch 114/200
4/4 [==============================] - 0s 998us/step - loss: 0.1277
Epoch 115/200
4/4 [=====================

Với mô hình được training, chúng ta có thể thấy mô hình đã phân loại dữ liệu training như thế nào.


In [7]:
plt_cat_mc(X_train, y_train, model, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Ở trên, ranh giới quyết định cho thấy mô hình đã phân vùng không gian đầu vào như thế nào.  Mô hình rất đơn giản này không gặp khó khăn gì trong việc phân loại dữ liệu training. Làm thế nào nó thực hiện được điều này? Hãy nhìn vào mạng chi tiết hơn. 

Dưới đây, chúng ta sẽ lấy các trọng số đã training từ mô hình và sử dụng trọng số đó để vẽ đồ thị chức năng của từng đơn vị mạng. Xa hơn nữa, có một lời giải thích chi tiết hơn về kết quả. Bạn không cần biết những chi tiết này để sử dụng thành công neural network, nhưng có thể hữu ích nếu bạn có thêm trực giác về cách các lớp kết hợp với nhau để giải quyết vấn đề phân loại.


In [8]:
# thu thập các tham số được training từ lớp đầu tiênl1 = model.get_layer("L1")
W1,b1 = l1.get_weights()

In [9]:
# vẽ sơ đồ chức năng của lớp đầu tiênplt_layer_relu(X_train, y_train.reshape(-1,), W1, b1, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [10]:
# thu thập các tham số được training từ lớp đầu ral2 = model.get_layer("L2")
W2, b2 = l2.get_weights()
# tạo ra các 'feature mới', các training example sau khi chuyển đổi L1Xl2 = np.maximum(0, np.dot(X_train,W1) + b1)

plt_output_layer_linear(Xl2, y_train.reshape(-1,), W2, b2, classes,
                        x0_rng = (-0.25,np.amax(Xl2[:,0])), x1_rng = (-0.25,np.amax(Xl2[:,1])))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Giải thích
#### Lớp 1 <img align="Right" src="./images/C2_W2_mclass_layer1.png"  style=" width:600px; padding: 10px 20px ; ">
Các sơ đồ này hiển thị chức năng của Đơn vị 0 và 1 trong lớp đầu tiên của mạng. Đầu vào là ($x_0,x_1$) trên trục. Đầu ra của thiết bị được thể hiện bằng màu nền. Điều này được biểu thị bằng thanh màu ở bên phải của mỗi biểu đồ. Lưu ý rằng vì các thiết bị này đang sử dụng ReLu nên đầu ra không nhất thiết nằm trong khoảng từ 0 đến 1 và trong trường hợp này lớn hơn 20 ở mức cao nhất. 
Các đường đồng mức trong biểu đồ này hiển thị điểm chuyển tiếp giữa đầu ra, $a^{[1]}_j$ bằng 0 và khác 0. Nhớ lại biểu đồ của ReLu :<img align="right" src="./images/C2_W2_mclass_relu.png"  style=" width:200px; padding: 10px 20px ; "> Đường đồng mức trong biểu đồ là điểm uốn trong ReLu.

Đơn vị 0 đã tách lớp 0 và 1 khỏi lớp 2 và 3. Các điểm ở bên trái dòng (lớp 0 và 1) sẽ xuất ra 0, trong khi các điểm ở bên phải sẽ xuất ra giá trị lớn hơn 0.  
Đơn vị 1 đã tách lớp 0 và 2 khỏi lớp 1 và 3. Các điểm phía trên dòng (lớp 0 và 2 ) sẽ tạo ra số 0, trong khi các điểm bên dưới sẽ tạo ra giá trị lớn hơn 0. Hãy xem điều này diễn ra như thế nào trong lớp tiếp theo!


#### Lớp 2, lớp đầu ra <img align="Right" src="./images/C2_W2_mclass_layer2.png"  style=" width:600px; padding: 10px 20px ; ">

Các dấu chấm trong các biểu đồ này là các training example được dịch bởi lớp đầu tiên. Một cách để nghĩ về điều này là lớp đầu tiên đã tạo ra một bộ feature mới để lớp thứ 2 đánh giá. Các trục trong các ô này là kết quả đầu ra của lớp $a^{[1]}_0$ và $a^{[1]}_1$ trước đó. Như dự đoán ở trên, lớp 0 và 1 (xanh dương và xanh lục) có $a^{[1]}_0 = 0$ trong khi lớp 0 và 2 (xanh lam và cam) có $a^{[1]}_1 = 0$.  
Một lần nữa, cường độ của màu nền biểu thị giá trị cao nhất.  
Đơn vị 0 sẽ tạo ra giá trị tối đa cho các giá trị gần (0,0), trong đó lớp 0 (màu xanh) đã được ánh xạ.    
Đơn vị 1 tạo ra giá trị cao nhất ở góc trên bên trái chọn loại 1 (màu xanh lá cây).  
Đơn vị 2 nhắm vào góc dưới bên phải nơi lớp 2 (màu cam) cư trú.  
Đơn vị 3 tạo ra các giá trị cao nhất ở phía trên bên phải khi chọn lớp cuối cùng của chúng ta (màu tím).  

Một khía cạnh khác không rõ ràng từ biểu đồ là các giá trị đã được phối hợp giữa các đơn vị. Việc một đơn vị tạo ra giá trị tối đa cho lớp mà nó đang chọn là chưa đủ mà nó còn phải là giá trị cao nhất trong tất cả các đơn vị để tính điểm trong lớp đó. Điều này được thực hiện bằng hàm softmax ngụ ý là một phần của hàm loss (`SparseCategoricalCrossEntropy`). Không giống như các chức năng kích hoạt khác, softmax hoạt động trên tất cả các đầu ra.

Bạn có thể sử dụng thành công neural network mà không cần biết chi tiết về mục đích của từng đơn vị. Hy vọng rằng ví dụ này đã cung cấp một số trực giác về những gì đang diễn ra.


## Chúc mừng!
Bạn đã học cách xây dựng và vận hành neural network để phân loại nhiều lớp.
